# Notes on LibXC/XCFun convention transformation

## Explanation by specific example

LibXC and XCFun have different conventions on how to handle the spin-polarized cases.

We will use GGA spin-polarized second-order derivative (fxc) as example to explain that.

As the first step, we state that for the energy part (zk) and first-order derivative part (vxc), LibXC and XCFun have the same conventions. The order is

| Index | Notation |
|--|--|
| 0 | zk |
| 1 | r_u |
| 2 | r_d |
| 3 | s_uu |
| 4 | s_ud |
| 5 | s_dd |

For the notation in above table,
- Rho is spin-polarization two-component, `r_u` means $\rho^\uparrow$, `r_d` means $\rho^\downarrow$.
- Sigma is spin-polarization three-component, `s_uu` means $\sigma^{\uparrow \uparrow}$, `s_ud` means $\sigma^{\uparrow \downarrow}$, `s_dd` means $\sigma^{\downarrow \downarrow}$.

For the fxc part, there will involve two densities as variables. The LibXC and XCFun have fundamental difference at

- LibXC respect which type of density first (sorted by rho, sigma), then its spin;
- XCFun respect the spin-component first (sorted by r_u, r_d, s_uu, s_ud, s_dd).

It is more favorable to use XCFun-style for future DFT evaluation. However, LibXC supports more functionals, with better API design and popularity. So an index transform mapping is required. For this specific task,

| Index | Notation<br>LibXC | Notation<br>XCFun | Map |
|-:|--|--|-:|
|  6 | r_u  / r_u  | r_u  / r_u  |  6 |
|  7 | r_u  / r_d  | r_u  / r_d  |  7 |
|  8 | r_d  / r_d  | r_u  / s_uu |  9 |
|  9 | r_u  / s_uu | r_u  / s_ud | 10 |
| 10 | r_u  / s_ud | r_u  / s_dd | 11 |
| 11 | r_u  / s_dd | r_d  / r_d  |  8 |
| 12 | r_d  / s_uu | r_d  / s_uu | 12 |
| 13 | r_d  / s_ud | r_d  / s_ud | 13 |
| 14 | r_d  / s_dd | r_d  / s_dd | 14 |
| 15 | s_uu / s_uu | s_uu / s_uu | 15 |
| 16 | s_uu / s_ud | s_uu / s_ud | 16 |
| 17 | s_uu / s_dd | s_uu / s_dd | 17 |
| 18 | s_ud / s_ud | s_ud / s_ud | 18 |
| 19 | s_ud / s_dd | s_ud / s_dd | 19 |
| 20 | s_dd / s_dd | s_dd / s_dd | 20 |

We can see that

- LibXC will first category `r/r` (6--8), then `r/s` (9--14), then `s/s` (16--20). For each category, sort by spin.
- XCFun will first category `r_u` (6--10), then `r_d` (11--14), then `s_uu` (15--17), then `s_ud` (18--19), then `s_dd` (20). For each category, sort by the same way in first category.

## Extension to the example

- **Higher derivative**: We may encounter more higher derivatives (usually up to 4th derivative, but can be more).
- **More kinds of density**: We may use more density types. The priority is RHO > SIGMA > TAU > LAPL.
  - TAU: `t_u`, `t_d`
  - LAPL: `l_u`, `l_d`
- We assume the RHO (LDA) only inputs RHO; SIGMA (GGA) inputs both RHO and SIGMA; TAU (some meta-GGA) inputs RHO SIGMA TAU, and LAPL (some meta-GGA) inputs all RHO SIGMA TAU LAPL (though some LAPL meta-GGAs does not actually input tau, but we still require the TAU to be available for simplicity).

## Task for Automatic Index Map Generation

We may have the following kind of function.

If everything goes smoothly, the GGA 3rd derivative should give output

```
[
    21, 22, 25, 26, 27, 23, 28, 29, 30, 34, 35, 36, 37, 38, 39, 24, 31, 32, 33, 40, 41, 42, 43, 44, 45, 46, 47,
    48, 49, 50, 51, 52, 53, 54, 55,
]
```

In [1]:
from itertools import combinations_with_replacement
from math import comb


def libxc_to_xcfun_indices_map(den_type: str, deriv: int) -> list[int]:
    """Spin-Polarized Indices Map from LibXC to XCFun.

    Parameters
    ----------
    den_type : str
        Density Type. Supports `rho`, `sigma`, `tau`, `lapl`.
    deriv : int
        Derivative level.

    Example
    -------
    >>> libxc_to_xcfun_indices_map("sigma", 2)
    [6, 7, 9, 10, 11, 8, 12, 13, 14, 15, 16, 17, 18, 19, 20]
    """
    # Each variable: (type_priority, spin_index)
    # RHO has 2 spin components, SIGMA has 3, TAU has 2, LAPL has 2
    group_specs = [
        ("rho", 0, 2),
        ("sigma", 1, 3),
        ("tau", 2, 2),
        ("lapl", 3, 2),
    ]

    type_map = {
        "rho": ["rho"],
        "sigma": ["rho", "sigma"],
        "tau": ["rho", "sigma", "tau"],
        "lapl": ["rho", "sigma", "tau", "lapl"],
    }

    if den_type not in type_map:
        raise ValueError(f"Unknown den_type: {den_type}")

    active_groups = set(type_map[den_type])

    # Build variable list: each variable is (type_priority, spin_index)
    variables = []
    for group_name, priority, n_spin in group_specs:
        if group_name in active_groups:
            for spin in range(n_spin):
                variables.append((priority, spin))

    d = len(variables)

    # Generate all non-decreasing multi-indices of length deriv
    # combinations_with_replacement yields them in lexicographic order = XCFun order
    xcfun_order = list(combinations_with_replacement(range(d), deriv))

    # LibXC order: sort by density type signature first, then by variable indices
    def libxc_key(mi):
        return tuple(variables[i][0] for i in mi) + mi

    libxc_order = sorted(xcfun_order, key=libxc_key)

    # Build reverse lookup: multi-index -> LibXC position
    libxc_pos = {mi: pos for pos, mi in enumerate(libxc_order)}

    # Base offset = sum of outputs for all previous derivative levels
    base_offset = sum(comb(d + i - 1, i) for i in range(deriv))

    return [base_offset + libxc_pos[mi] for mi in xcfun_order]

# Actual Output

In [2]:
print(libxc_to_xcfun_indices_map("sigma", 2))

[6, 7, 9, 10, 11, 8, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [3]:
print(libxc_to_xcfun_indices_map("sigma", 3))

[21, 22, 25, 26, 27, 23, 28, 29, 30, 34, 35, 36, 37, 38, 39, 24, 31, 32, 33, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]


In [4]:
print(libxc_to_xcfun_indices_map("sigma", 4))

[56, 57, 61, 62, 63, 58, 64, 65, 66, 73, 74, 75, 76, 77, 78, 59, 67, 68, 69, 79, 80, 81, 82, 83, 84, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 60, 70, 71, 72, 85, 86, 87, 88, 89, 90, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125]


In [5]:
print(libxc_to_xcfun_indices_map("tau", 2))

[8, 9, 11, 12, 13, 17, 18, 10, 14, 15, 16, 19, 20, 21, 22, 23, 27, 28, 24, 25, 29, 30, 26, 31, 32, 33, 34, 35]


In [6]:
print(libxc_to_xcfun_indices_map("tau", 3))

[36, 37, 40, 41, 42, 49, 50, 38, 43, 44, 45, 51, 52, 55, 56, 57, 67, 68, 58, 59, 69, 70, 60, 71, 72, 79, 80, 81, 39, 46, 47, 48, 53, 54, 61, 62, 63, 73, 74, 64, 65, 75, 76, 66, 77, 78, 82, 83, 84, 85, 86, 87, 95, 96, 88, 89, 97, 98, 90, 99, 100, 107, 108, 109, 91, 92, 101, 102, 93, 103, 104, 110, 111, 112, 94, 105, 106, 113, 114, 115, 116, 117, 118, 119]


In [7]:
print(libxc_to_xcfun_indices_map("tau", 4))

[120, 121, 125, 126, 127, 137, 138, 122, 128, 129, 130, 139, 140, 145, 146, 147, 163, 164, 148, 149, 165, 166, 150, 167, 168, 181, 182, 183, 123, 131, 132, 133, 141, 142, 151, 152, 153, 169, 170, 154, 155, 171, 172, 156, 173, 174, 184, 185, 186, 190, 191, 192, 210, 211, 193, 194, 212, 213, 195, 214, 215, 234, 235, 236, 196, 197, 216, 217, 198, 218, 219, 237, 238, 239, 199, 220, 221, 240, 241, 242, 252, 253, 254, 255, 124, 134, 135, 136, 143, 144, 157, 158, 159, 175, 176, 160, 161, 177, 178, 162, 179, 180, 187, 188, 189, 200, 201, 202, 222, 223, 203, 204, 224, 225, 205, 226, 227, 243, 244, 245, 206, 207, 228, 229, 208, 230, 231, 246, 247, 248, 209, 232, 233, 249, 250, 251, 256, 257, 258, 259, 260, 261, 262, 275, 276, 263, 264, 277, 278, 265, 279, 280, 295, 296, 297, 266, 267, 281, 282, 268, 283, 284, 298, 299, 300, 269, 285, 286, 301, 302, 303, 313, 314, 315, 316, 270, 271, 287, 288, 272, 289, 290, 304, 305, 306, 273, 291, 292, 307, 308, 309, 317, 318, 319, 320, 274, 293, 294, 310, 311,